# Assignment 1 – Problem 3
## Satellite Image Classification (Urban vs. Natural) Using Classical Vision + ML  
### Classical Computer Vision + Machine Learning
**Course:** Computer Vision (S2-25_AIMLCZG525)

---

### Goal 
Build a classical machine-learning system that classifies satellite images into Urban or 
Natural categories using handcrafted image features. The system must incorporate pixel
level, texture, and gradient-based features and evaluate performance on a real external 
image. 

### Implementation Overview
1. Dataset Preparation (fMoW dataset – Urban vs Natural, ~500 each)
2. Preprocessing (resize, convert to grayscale, CLAHE, Gaussian blur)
3. Feature Engineering (Pixel intensity histogram, LBP texture descriptor, Edges using Sobel/Canny, Histogram of Oriented Gradients (HOG))
4. Feature Selection (PCA)
5. Model Training (SVM + Random Forest, 5-fold CV)
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. External Image Prediction
8. Analysis & Discussion

## Import Required Libraries

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
import warnings
import pickle
#warnings.filterwarnings('ignore')

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import kagglehub

# ── Image Processing ──────────────────────────────────────────────────────────
import cv2                                      # OpenCV for image I/O and processing
from skimage.feature import hog, local_binary_pattern  # HOG and LBP
from skimage import exposure

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ All libraries imported successfully.")
print(f"   OpenCV   : {cv2.__version__}")
print(f"   NumPy    : {np.__version__}")
print(f"   Pandas   : {pd.__version__}")

## 1. Dataset Preparation
**fMoW Dataset – Urban vs Natural (~500 images each)**

- Download and organize fMoW dataset images into two classes: Urban and Natural
- Limit to ~500 images per class for balanced training
- Split into train/validation/test sets

In [ ]:
# 1. Dataset Preparation
EXTERNAL_IMG   = 'external_test.jpg'   # your own real-world image
IMG_SIZE       = (128, 128)
MAX_PER_CLASS  = 500             # cap to keep runtime manageable
FEATURES_FILE  = 'features.pkl'  # cached features

# Define urban and natural classes for EuroSAT
# Mapping EuroSAT classes to broader Urban/Natural categories.
# EuroSAT classes generally include: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture,
# PermanentCrop, Residential, River, SeaLake.
URBAN_EUROSAT_CATEGORIES = ['Residential', 'Industrial', 'Highway']
NATURAL_EUROSAT_CATEGORIES = ['Forest', 'River', 'SeaLake', 'Pasture', 'PermanentCrop', 'HerbaceousVegetation', 'AnnualCrop']

# Download the EuroSAT dataset from Kaggle
print("Downloading EuroSAT dataset from Kaggle... This may take a moment.")
# The path returned by dataset_download is typically the base directory of the extracted dataset
try:
    eurosat_path = kagglehub.dataset_download("waseemalastal/eurosat-rgb-dataset")
    print(f"EuroSAT dataset downloaded to: {eurosat_path}")
except Exception as e:
    print(f"Error downloading from Kaggle: {e}")
    print("Please ensure you have authenticated Kaggle API if necessary (e.g., using `kaggle.json`).")
    eurosat_path = None # Set to None to prevent further processing if download fails

all_images = []
all_labels = []

urban_count = 0
natural_count = 0

if eurosat_path:
    # Iterate through the downloaded dataset directory to collect images
    # Assuming the structure is eurosat_path/class_name/image.jpg
    for root, dirs, files in os.walk(eurosat_path):
        # Extract the class name from the current directory
        # The class folder is usually directly under the main dataset folder
        if root == eurosat_path:
            continue # Skip the root directory itself, which only contains class directories

        class_name = os.path.basename(root)
        print(f"Processing class folder: {class_name}")

        target_label = None
        if class_name in URBAN_EUROSAT_CATEGORIES:
            target_label = 'urban'
        elif class_name in NATURAL_EUROSAT_CATEGORIES:
            target_label = 'natural'
        else:
            print(f"  Skipping class '{class_name}' as it's not mapped to urban/natural.")
            continue

        # Check if we still need images for this target label
        if target_label == 'urban' and urban_count >= MAX_PER_CLASS:
            print(f"  Already collected {MAX_PER_CLASS} images for 'urban'. Skipping this class.")
            continue
        if target_label == 'natural' and natural_count >= MAX_PER_CLASS:
            print(f"  Already collected {MAX_PER_CLASS} images for 'natural'. Skipping this class.")
            continue

        for file_name in files:
            if urban_count >= MAX_PER_CLASS and natural_count >= MAX_PER_CLASS:
                break # Collected enough images for both categories overall

            # Only process image files (e.g., .jpg, .png)
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(root, file_name)
                img_cv2 = cv2.imread(image_path)

                if img_cv2 is None:
                    print(f"  Warning: Could not read image {image_path}. Skipping.")
                    continue

                # cv2.imread reads as BGR by default for color images. If it's grayscale, it's 2D.
                # Convert to BGR if needed (e.g., if it's grayscale or RGBA, for consistency).
                if img_cv2.ndim == 2: # Grayscale image
                    img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_GRAY2BGR)
                elif img_cv2.shape[2] == 4: # RGBA image
                    img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_RGBA2BGR)


                if target_label == 'urban' and urban_count < MAX_PER_CLASS:
                    all_images.append(img_cv2)
                    all_labels.append('urban')
                    urban_count += 1
                elif target_label == 'natural' and natural_count < MAX_PER_CLASS:
                    all_images.append(img_cv2)
                    all_labels.append('natural')
                    natural_count += 1

        if urban_count >= MAX_PER_CLASS and natural_count >= MAX_PER_CLASS:
            print(f"  Reached MAX_PER_CLASS for both 'urban' ({urban_count}) and 'natural' ({natural_count}). Stopping collection.")
            break # Exit outer loop if enough images are collected
else:
    print("Dataset download failed or path not found. Cannot proceed with image loading.")

print(f"\nTotal images loaded : {len(all_images)}")
print(f"  Urban   : {all_labels.count('urban')}")
print(f"  Natural : {all_labels.count('natural')}")

Error downloading from Kaggle: name 'kagglehub' is not defined
Please ensure you have authenticated Kaggle API if necessary (e.g., using `kaggle.json`).
Dataset download failed or path not found. Cannot proceed with image loading.

Total images loaded : 0
  Urban   : 0
  Natural : 0


## 2. Preprocessing
**Steps: Resize → Grayscale → CLAHE → Gaussian Blur**

- Resize all images to a fixed resolution (e.g., 128×128)
- Convert to grayscale for uniform feature extraction
- Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
- Apply Gaussian blur to reduce noise

In [ ]:
# 2. Preprocessing
# ── CLAHE object (reused across images) ───────────────────────────────────────
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_image(img, size=IMG_SIZE):
    """
    Full preprocessing pipeline:
      1. Resize to fixed size
      2. Convert BGR → Grayscale
      3. CLAHE for adaptive contrast enhancement
      4. Gaussian Blur (3x3) for noise reduction
    Returns preprocessed grayscale image (uint8).
    """
    img_resized = cv2.resize(img, size)                       # Step 1: Resize
    img_gray    = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)  # Step 2: Grayscale
    img_clahe   = clahe.apply(img_gray)                       # Step 3: CLAHE
    img_blur    = cv2.GaussianBlur(img_clahe, (3, 3), 0)      # Step 4: Gaussian Blur
    return img_blur

# ── Apply preprocessing to all images ─────────────────────────────────────────
print("Preprocessing images …")
preprocessed = [preprocess_image(img) for img in all_images]
print(f"Done. Shape of first preprocessed image: {preprocessed[0].shape}")

In [ ]:
# ── Preprocessing Visualization ───────────────────────────────────────────────
sample_idx = 0   # show steps for the first image
orig  = all_images[sample_idx]

orig_rgb   = cv2.cvtColor(cv2.resize(orig, IMG_SIZE), cv2.COLOR_BGR2RGB)
gray       = cv2.cvtColor(cv2.resize(orig, IMG_SIZE), cv2.COLOR_BGR2GRAY)
cl         = clahe.apply(gray)
blurred    = cv2.GaussianBlur(cl, (3, 3), 0)

steps = [orig_rgb, gray, cl, blurred]
titles = ['Original (RGB)', 'Grayscale', 'CLAHE', 'Gaussian Blur']
cmaps  = [None, 'gray', 'gray', 'gray']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Preprocessing Pipeline – Step by Step', fontsize=14, fontweight='bold')
for ax, step, title, cmap in zip(axes, steps, titles, cmaps):
    ax.imshow(step, cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig('preprocessing_steps.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering
**Descriptors: Pixel Intensity Histogram | LBP | Sobel/Canny Edges | HOG**

- **Pixel Intensity Histogram**: Distribution of pixel values per channel
- **LBP (Local Binary Pattern)**: Texture descriptor capturing local structure
- **Edges (Sobel / Canny)**: Edge magnitude maps for structural features
- **Canny Edges**: Edge density
- **HOG (Histogram of Oriented Gradients)**: Shape and gradient orientation features

In [ ]:
# 3. Feature Engineering
# ─────────────────────────────────────────────────────────────────────────────
# Feature Extraction Functions
# ─────────────────────────────────────────────────────────────────────────────
import cv2
from skimage.feature import hog, local_binary_pattern
from skimage import exposure

def extract_all_features(img):
    """
    Concatenate all feature vectors into one feature vector per image.
    Order: [Intensity Hist | LBP | Sobel Edges | Canny Density | HOG]
    """
    
    # Global pixel intensity histogram captures brightness distribution
    bins = 256
    hist = cv2.calcHist([img], [0], None, [bins], [0, 256])
    hist = hist.flatten() / (img.shape[0] * img.shape[1])  # normalize by pixel count


    # Local Binary Pattern (LBP) texture descriptor
    num_circular_neighbours = 8, radius = 1
    lbp  = local_binary_pattern(img, P=num_circular_neighbours, R=radius, method='uniform')
    n_bins = int(lbp.max() + 1)
    lbp_hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins), density=True)

    # Sobel gradient magnitude histogram captures structural gradients
    sobel_x  = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y  = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    magnitude = (magnitude / magnitude.max() * 255).astype(np.uint8)
    sobel_hist = cv2.calcHist([magnitude], [0], None, [256], [0, 256])
    sobel_hist = hist.flatten() / magnitude.size


    # Canny edge density – single scalar: fraction of edge pixels
    edges = cv2.Canny(img, threshold1=50, threshold2=150)
    density = np.sum(edges > 0) / edges.size
    canny = np.array([density])   # shape: (1,)

    # Histogram of Oriented Gradients (HOG).
    # For 128×128 image: output dim = 9 * (16*16) * 1 = 2304 (approx).
    hist_of_grads = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False,
        feature_vector=True
    )

    return np.concatenate([hist, lbp_hist, sobel_hist, canny, hist_of_grads])


print("Try extract_all_features on one image.")
extract_feature_test = extract_all_features(preprocessed[0])
print(f"Feature vector length for one image: {len(extract_feature_test)}")


# Extract features for all images
print("Extracting features … (may take a few minutes)")
X = np.array([extract_all_features(img) for img in preprocessed])
y = np.array(all_labels)

print(f"\nFeature matrix shape : {X.shape}")
print(f"Labels shape         : {y.shape}")
print(f"Classes              : {np.unique(y)}")


In [ ]:
# Feature Visualization
# Show HOG visualization and Sobel/Canny on a sample image
sample  = preprocessed[0]
label_s = all_labels[0]

# LBP image
lbp_img  = local_binary_pattern(sample, P=8, R=1, method='uniform')

# Sobel
sobel_x  = cv2.Sobel(sample, cv2.CV_64F, 1, 0, ksize=3)
sobel_y  = cv2.Sobel(sample, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_mag = (sobel_mag / sobel_mag.max() * 255).astype(np.uint8)

# Canny
canny    = cv2.Canny(sample, 50, 150)

# HOG visualization
hog_feat, hog_img = hog(
    sample, orientations=9, pixels_per_cell=(8, 8),
    cells_per_block=(2, 2), visualize=True, feature_vector=True
)
hog_img_rescaled = exposure.rescale_intensity(hog_img, in_range=(0, 10))


fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f'Feature Visualizations Label: {label_s}', fontsize=14, fontweight='bold')

axes[0].set_title('Preprocessed')
axes[0].imshow(sample, cmap='gray')

axes[1].set_title('LBP')
axes[1].imshow(lbp_img, cmap='gray')

axes[2].set_title('Sobel Edges')
axes[2].imshow(sobel_mag, cmap='gray')

axes[3].set_title('Canny Edges')
axes[3].imshow(canny, cmap='gray')

axes[4].set_title('HOG')
axes[4].imshow(hog_img_rescaled, cmap='gray')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Feature Selection
**Method: Principal Component Analysis (PCA)**

- Concatenate all engineered features into a single feature vector
- Apply PCA to reduce dimensionality while retaining variance (e.g., 95%)
- Visualize explained variance ratio

In [ ]:
# 4. Feature Selection (PCA)
# Train / Test Split (80:20 stratified)
lbl_enc = LabelEncoder()
y_enc = lbl_enc.fit_transform(y)   # urban=1, natural=0 (alphabetical)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=SEED, stratify=y_enc
)
print(f"Train size : {X_train.shape[0]}   Test size : {X_test.shape[0]}")

# StandardScaler (Z-score normalization)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Principal Component Analysis
pca = PCA(n_components=0.95, random_state=SEED)   # retain 95% variance
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

print(f"Original feature dimension : {X_train_sc.shape[1]}")
print(f"PCA reduced dimension with 95% variance retained      : {X_train_pca.shape[1]}")

In [ ]:
# Plot PCA Cumulative Explained Variance
cumvar = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(cumvar, color='#2196F3', linewidth=2)
plt.axhline(0.95, color='red', linestyle='--', label='95% threshold')
plt.axvline(pca.n_components_, color='green', linestyle='--',
            label=f'{pca.n_components_} components')
plt.fill_between(range(len(cumvar)), cumvar, alpha=0.15, color='#2196F3')
plt.xlabel('Number of Principal Components', fontsize=12)
plt.ylabel('Cumulative Explained Variance', fontsize=12)
plt.title('PCA Cumulative Explained Variance', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"PCA retains {pca.n_components_} components out of {X_train_sc.shape[1]}")

## 5. Model Training
**Classifiers: SVM + Random Forest | Validation: 5-Fold Cross-Validation**

- Train Support Vector Machine (SVM) with RBF kernel on PCA-reduced features
- Train Random Forest classifier with tuned hyperparameters
- Apply 5-fold cross-validation to assess generalization performance

In [ ]:
# 5. Model Training (SVM + Random Forest, 5-Fold CV)

# ──────────────────────────────────────────────────────────────
# Define four classical ML classifiers as required by the assignment
# ──────────────────────────────────────────────────────────────
classifiers = {
    'k-NN (k=5)': KNeighborsClassifier(
        n_neighbors=5,
        metric='euclidean',
        n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf',
        C=10,
        gamma='scale',
        decision_function_shape='ovr',
        random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        n_jobs=-1,
        random_state=SEED
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=500,
        solver='lbfgs',
        multi_class='multinomial',
        C=1.0,
        random_state=SEED
    )
}

print('Classifiers defined:')
for name in classifiers:
    print(f'   • {name}')


# ── 5-fold Stratified Cross-Validation on training set ─────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = {}
print('Running 5-fold cross-validation …\n')

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_tr, y_train,
                              cv=skf, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:<25} CV Accuracy: '
          f'{scores.mean():.4f} ± {scores.std():.4f}')

# ── CV Results Bar Chart ─────────────────────────────────────
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=['#4C72B0','#DD8452','#55A868','#C44E52'],
              edgecolor='black', linewidth=0.6)
ax.set_ylim(0, 1.0)
ax.set_ylabel('CV Accuracy', fontsize=11)
ax.set_title('5-Fold Cross-Validation Accuracy per Model', fontweight='bold')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.01,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


import time

trained_models = {}
train_times    = {}

print('Training classifiers on full training set …\n')
for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_tr, y_train)
    elapsed = time.time() - t0
    trained_models[name] = clf
    train_times[name]    = elapsed
    print(f'   {name:<25} trained in {elapsed:.2f}s')

print('\nAll models training completed.')

## 6. Evaluation
**Metrics: Accuracy | Precision | Recall | F1-Score | Confusion Matrix**

- Compute Accuracy, Precision, Recall, and F1-Score for each model
- Plot Confusion Matrix for SVM and Random Forest
- Compare performance of both classifiers

In [ ]:
# 6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)

eval_results = {}

print(f'{'Model':<25} {'Accuracy':>10} {'Precision':>11} {'Recall':>8} {'F1-Score':>10}')
print('-' * 70)

for name, clf in trained_models.items():
    y_pred = clf.predict(X_te)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    eval_results[name] = {
        'accuracy' : acc,
        'precision': prec,
        'recall'   : rec,
        'f1'       : f1,
        'y_pred'   : y_pred
    }
    print(f'{name:<25} {acc:>10.4f} {prec:>11.4f} {rec:>8.4f} {f1:>10.4f}')

# ── Best model ──────────────────────────────────────────────
best_name = max(eval_results, key=lambda k: eval_results[k]['f1'])
print(f'\nBest model (by F1): {best_name}  '
      f'(F1 = {eval_results[best_name]["f1"]:.4f})')


## 7. External Image Prediction
**Predict class (Urban / Natural) for a new unseen image**

- Load an external satellite image not seen during training
- Apply the same preprocessing and feature extraction pipeline
- Predict and display the class label using the best trained model

In [ ]:
# 7. External Image Prediction
# ── Load and Predict External Image ───────────────────────────────────────────
if not os.path.exists(EXTERNAL_IMG):
    print(f"⚠️  External image '{EXTERNAL_IMG}' not found.")
    print("   Please place your image in the working directory and re-run this cell.")
else:
    ext_img_raw = cv2.imread(EXTERNAL_IMG)
    ext_prep    = preprocess_image(ext_img_raw)
    ext_feat    = extract_all_features(ext_prep).reshape(1, -1)
    ext_feat_sc = scaler.transform(ext_feat)
    ext_feat_pca= pca.transform(ext_feat_sc)

    # Display preprocessing steps
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle('External Image – Preprocessing Pipeline', fontsize=14, fontweight='bold')
    orig_disp = cv2.cvtColor(cv2.resize(ext_img_raw, IMG_SIZE), cv2.COLOR_BGR2RGB)
    gray_e    = cv2.cvtColor(cv2.resize(ext_img_raw, IMG_SIZE), cv2.COLOR_BGR2GRAY)
    clahe_e   = clahe.apply(gray_e)
    blur_e    = cv2.GaussianBlur(clahe_e, (3,3), 0)
    for ax, img, title, cmap in zip(
        axes,
        [orig_disp, gray_e, clahe_e, blur_e],
        ['Original', 'Grayscale', 'CLAHE', 'Blurred'],
        [None, 'gray', 'gray', 'gray']
    ):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Predict
    print("\n── External Image Predictions ──")
    for name, model in trained_models.items():
        pred_enc  = model.predict(ext_feat_pca)[0]
        pred_lbl  = le.inverse_transform([pred_enc])[0]
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(ext_feat_pca)[0].max() * 100
            print(f"  {name:<20s} → {pred_lbl.upper():>8s}  (confidence: {prob:.1f}%)")
        else:
            print(f"  {name:<20s} → {pred_lbl.upper():>8s}")

    # Visual result card
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(cv2.cvtColor(cv2.resize(ext_img_raw, IMG_SIZE), cv2.COLOR_BGR2RGB))
    preds_str = '  |  '.join(
        f"{n.split()[0]}: {le.inverse_transform([m.predict(ext_feat_pca)[0]])[0].upper()}"
        for n, m in trained_models.items()
    )
    ax.set_title(f'External Image Prediction\n{preds_str}',
                 fontsize=10, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('external_prediction.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Analysis & Discussion
**Observations, Insights, and Conclusions**

- Discuss which features contributed most to classification performance
- Compare SVM vs Random Forest: strengths and weaknesses
- Analyze where the model succeeds and where it fails (error analysis)
- Suggest possible improvements (e.g., deep features, data augmentation)
- Final conclusions and takeaways

In [ ]:
# 8. Analysis & Discussion
# ── Final Summary Table ───────────────────────────────────────────────────────
summary_data = []
for name, r in results.items():
    summary_data.append({
        'Model'    : name,
        'Accuracy' : round(r['accuracy'], 4),
        'Precision': round(r['precision'], 4),
        'Recall'   : round(r['recall'], 4),
        'F1-Score' : round(r['f1'], 4),
        'CV Mean'  : round(cv_results[name].mean(), 4),
        'CV Std'   : round(cv_results[name].std(), 4),
    })

summary_df = pd.DataFrame(summary_data).set_index('Model')
print("\n" + "="*60)
print(" FINAL RESULTS SUMMARY")
print("="*60)
print(summary_df.to_string())
print("="*60)
summary_df.to_csv('results_summary.csv')
print("\nResults saved → results_summary.csv")
